# geo-resolver — Visual Test Notebook

Resolve natural language queries and visualize them on an interactive map.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import geopandas as gpd
import contextily as cx
from shapely.geometry import shape
from dotenv import load_dotenv

load_dotenv()

from geo_resolver import GeoResolver

# OpenRouter pricing for Gemini 3 Flash (per 1M tokens)
PRICING = {"input": 0.0005 / 1000, "output": 0.003 / 1000}

# Basemap style — Carto Voyager (labels, clean, free)
BASEMAP = cx.providers.CartoDB.Voyager

def estimate_cost(usage):
    return usage.prompt_tokens * PRICING["input"] + usage.completion_tokens * PRICING["output"]

def print_steps(steps):
    for step in steps:
        if step.get("type") == "thinking":
            print(f"  💭 {step['message'][:120]}")
        elif step.get("type") == "cache_hit":
            print(f"  ⚡ {step['message']}")
        else:
            print(f"  🔧 {step.get('message', step.get('tool', '?'))}")

def resolve_and_map(query: str, padding=0.5, figsize=(10, 7), **kwargs):
    """Resolve a query and render on a static map with real basemap tiles."""
    with GeoResolver(**kwargs) as resolver:
        result = resolver.resolve(query)
    
    geojson = result.geojson
    geom = shape(geojson["geometry"])
    gdf = gpd.GeoDataFrame([{"geometry": geom}], crs="EPSG:4326")
    
    display_name = result.resolved_name or query
    
    # Print info
    print(f"Query: {query}")
    if result.resolved_name and result.resolved_name.lower() != query.lower():
        print(f"Resolved: {result.resolved_name}")
    print(f"Steps ({len(result.steps)}):")
    print_steps(result.steps)
    cost = estimate_cost(result.usage)
    area = gdf.to_crs(epsg=3857).area.iloc[0] / 1e6
    print(f"\nTokens: {result.usage.prompt_tokens:,} in / {result.usage.completion_tokens:,} out")
    print(f"Cost: ${cost:.6f}")
    print(f"Geometry: {geom.geom_type} — {area:.1f} km²")
    
    # Reproject to Web Mercator for tile alignment
    gdf_wm = gdf.to_crs(epsg=3857)
    minx, miny, maxx, maxy = gdf_wm.total_bounds
    dx = (maxx - minx) or 1000
    dy = (maxy - miny) or 1000
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Draw geometry
    gdf_wm.plot(ax=ax, facecolor="#3388ff44", edgecolor="#2255cc", linewidth=2)
    
    # Set view with padding
    ax.set_xlim(minx - dx * padding, maxx + dx * padding)
    ax.set_ylim(miny - dy * padding, maxy + dy * padding)
    
    # Add basemap tiles (cached after first download)
    cx.add_basemap(ax, source=BASEMAP)
    
    # Label the result
    centroid_wm = gdf_wm.geometry.iloc[0].centroid
    ax.text(centroid_wm.x, centroid_wm.y, display_name,
            fontsize=11, ha="center", va="center",
            fontweight="bold", color="#1a1a2e",
            path_effects=[pe.withStroke(linewidth=3, foreground="white")])
    
    ax.set_axis_off()
    
    title = f'"{query}"'
    if result.resolved_name and result.resolved_name.lower() != query.lower():
        title += f"  →  {result.resolved_name}"
    ax.set_title(title, fontsize=12, color="#333333", pad=10)
    
    plt.tight_layout()
    plt.show()
    
    return result

## Test: San Francisco

In [ ]:
resolve_and_map("San Francisco")

## Test: Try your own queries

Edit the query below and run the cell.

In [ ]:
resolve_and_map("The city that Taj Mahal is located in")

In [ ]:
resolve_and_map("Northern California")

In [ ]:
# Compare multiple results on one map
queries = ["San Francisco", "Oakland", "San Jose"]
colors = ["#e41a1c88", "#377eb888", "#4daf4a88"]
edge_colors = ["#e41a1c", "#377eb8", "#4daf4a"]

results = []
with GeoResolver() as resolver:
    for query in queries:
        r = resolver.resolve(query)
        geom = shape(r.geojson["geometry"])
        name = r.resolved_name or query
        results.append({"query": query, "name": name, "geometry": geom})

all_gdf = gpd.GeoDataFrame(results, crs="EPSG:4326").to_crs(epsg=3857)
minx, miny, maxx, maxy = all_gdf.total_bounds
dx = (maxx - minx) or 1000
dy = (maxy - miny) or 1000

fig, ax = plt.subplots(figsize=(10, 8))

for i, (row, fc, ec) in enumerate(zip(results, colors, edge_colors)):
    gdf = gpd.GeoDataFrame([row], crs="EPSG:4326").to_crs(epsg=3857)
    gdf.plot(ax=ax, facecolor=fc, edgecolor=ec, linewidth=2)
    c = gdf.geometry.iloc[0].centroid
    ax.text(c.x, c.y, row["name"], fontsize=9, ha="center", va="center",
            fontweight="bold", color=ec,
            path_effects=[pe.withStroke(linewidth=2, foreground="white")])

ax.set_xlim(minx - dx*0.3, maxx + dx*0.3)
ax.set_ylim(miny - dy*0.3, maxy + dy*0.3)
cx.add_basemap(ax, source=BASEMAP)
ax.set_axis_off()
ax.set_title("Bay Area Cities", fontsize=12, color="#333333", pad=10)
plt.tight_layout()
plt.show()